# Post 3: a ViT on Imagenette, on Colab

Trains the Vision Transformer in `model.py` on Imagenette at 160x160, with post 1's
recipe, so it compares directly with post 1's MLP and ResNet. Every setting is in
`train.RUNS`, and this notebook only runs them.

On Colab, checkpoints go straight to a GCS bucket and metrics go to Weights & Biases, so
both outlive the runtime. **If the runtime goes away, start a new one and run the
notebook again: each run carries on from its newest checkpoint in the bucket, and a
finished run only evaluates.**

Run locally, it trains the `smoke` run with outputs under `outputs/`, which checks the
whole notebook in seconds. `COLAB.md` explains how to run it on Colab with
`colab exec`.

## Setup

On Colab this clones the branch below and installs it. `BRANCH` must already be pushed,
because the runtime sees GitHub, not your working copy. Locally it does nothing.

The repo needs a JAX recent enough to have `jax.set_mesh`. If the runtime's is older,
this upgrades it before anything imports JAX.

In [ ]:
import os
import pathlib
import sys

BRANCH = "p03-vit"

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/toylib").exists():
        !git clone --depth 1 --branch {BRANCH} https://github.com/anujkhare/toylib.git /content/toylib
    %cd /content/toylib
    !git pull --ff-only origin {BRANCH}
    !pip install -q -e ".[harness,viz,gcs]"
    has_set_mesh = !python -c "import jax; print(hasattr(jax, 'set_mesh'))"
    if has_set_mesh[-1] != "True":
        !pip install -q -U "jax[tpu]"

## Sign in to GCS and Weights & Biases

`colab exec` can't answer prompts, so this cell asks for nothing. `COLAB.md` shows how a
service-account key and the W&B key reach the runtime as files. Neither goes through
`colab exec --env`, which saves values in plain text in the CLI's local history.

If the runtime can't read the bucket, this stops here rather than failing hours later.

In [ ]:
WANDB_PROJECT = "toylib-p03-vit"
WANDB_ENTITY = "anujkhare18"
BUCKET = "gs://tinystories-checkpoints"
CHECKPOINT_ROOT = f"{BUCKET}/toylib-p03-vit"
WANDB_KEY_FILE = pathlib.Path("/content/.wandb_key")

if IN_COLAB:
    project = os.environ.get("GOOGLE_CLOUD_PROJECT")
    if project:
        !gcloud config set project {project}
    bucket_access = !gcloud storage ls {BUCKET}/ 2>&1 | head -1
    if not bucket_access or not bucket_access[0].startswith("gs://"):
        raise RuntimeError(f"Can't read the bucket, so check the service-account key; see COLAB.md: {bucket_access}")
    if "WANDB_API_KEY" not in os.environ:
        if not WANDB_KEY_FILE.exists():
            raise RuntimeError(f"Upload the W&B key to {WANDB_KEY_FILE} first; see COLAB.md.")
        os.environ["WANDB_API_KEY"] = WANDB_KEY_FILE.read_text().strip()

## The runs

`RUNS_TO_TRAIN` picks runs from `train.RUNS` by name, separated by commas. It defaults to
`vit_ti16` on Colab and `smoke` locally. A run's name and epochs name its bucket folder
and its W&B run, so reusing a name resumes that run. Give a changed experiment a new name.

In [ ]:
import dataclasses

import jax
from absl import logging as absl_logging

from toylib_projects.p03_vit import train

RUNS_TO_TRAIN = os.environ.get("RUNS_TO_TRAIN", "vit_ti16" if IN_COLAB else "smoke").split(",")
unknown = sorted(set(RUNS_TO_TRAIN) - set(train.RUNS))
if unknown:
    raise ValueError(f"Unknown runs {unknown}; choose from {sorted(train.RUNS)}")

absl_logging.set_verbosity(absl_logging.ERROR)  # Quiet orbax's read-only notices.
print("JAX", jax.__version__, "| devices:", jax.devices())
if IN_COLAB and jax.default_backend() == "cpu":
    raise RuntimeError("No accelerator on this runtime, so training would take days.")


def destination(name):
    """On Colab, the bucket and W&B. Locally, the defaults under outputs/."""
    if not IN_COLAB:
        return {}
    return {
        "checkpoint_dir": f"{CHECKPOINT_ROOT}/{train.run_id(name, train.RUNS[name])}",
        "logger_config": train.wandb_logger(name, WANDB_PROJECT, WANDB_ENTITY),
    }


for name in RUNS_TO_TRAIN:
    print(f"\n{name}:")
    for field, value in dataclasses.asdict(train.RUNS[name]).items():
        print(f"  {field}: {value}")

## How long will this take?

A short timing run, saved nowhere: 50 training steps and one evaluation after compiling.
It also proves that the model, sharding and evaluation work on this accelerator before
hours go into them. A step here is the same size as a real step, so the estimate covers
the whole run, including one evaluation per epoch.

In [ ]:
TIMING_STEPS = 50
estimates = {}
for name in RUNS_TO_TRAIN:
    timing = train.time_run(name, steps=TIMING_STEPS)
    estimates[name] = timing["hours"]
    print(f"{name:12s} {timing['params']:>10,} params  {timing['steps_per_second']:6.1f} steps/s  "
          f"{timing['seconds_per_eval']:5.1f} s/eval  about {timing['hours']:.2f} h")
print(f"total: about {sum(estimates.values()):.2f} h")

## Train

On Colab this prints a W&B link, where the curves appear as it trains. Each run starts
from its newest checkpoint if it has one, and saves one every
`checkpoint_every_epochs`. The harness drops a partial batch, so the evaluation during
training misses a few test images, and `full_test_accuracy` covers all of them.

In [ ]:
results = {}
for name in RUNS_TO_TRAIN:
    results[name] = train.train(name, **destination(name))
    print(f"{name}: test accuracy {results[name]['full_test_accuracy']:.4f}, "
          f"test loss {results[name]['val/loss']:.4f}")

## Results

Post 1 reached 58.1% with its MLP and 87.4% with its ResNet on the same test images. The
checkpoints stay in the bucket under `CHECKPOINT_ROOT`.

In [ ]:
for name, result in results.items():
    print(f"{name:12s} test accuracy {result['full_test_accuracy']:.4f}  test loss {result['val/loss']:.4f}")